# NB-13: Audio Encoding Pathway -- CausalConv1d, MotionEncoder_tc, CausalAudioEncoder

## Learning Objectives

1. **Understand CausalConv1d's asymmetric padding** that preserves causality -- pad past, not future -- so each output timestep depends only on current and past inputs.
2. **Trace MotionEncoder_tc's dual local/global paths** showing how multi-head local features and single-head global features are produced with 4x temporal downsampling.
3. **See how CausalAudioEncoder performs weighted aggregation** over WAV2Vec layers via learnable SiLU-gated weights before routing through MotionEncoder_tc.

## Prerequisites

- **NB-01** (RMSNorm, basic ops) -- foundational normalization concepts
- **NB-08** (CausalConv3d analogy) -- same causal padding concept applied in 3D for the VAE; NB-13 mirrors this for 1D temporal convolutions in the S2V audio path

## Concept Map

```
WAV2Vec features  -->  CausalAudioEncoder (weighted aggregation)
                           |
                           v
                  MotionEncoder_tc (local + global paths)
                           |
                           v
                  Audio embeddings consumed by AudioInjector (NB-15)
```

In [ ]:
# Source: wan_video_dit_s2v.py -- S2V module loading setup
import sys, types, importlib.util, pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for audio encoding demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py (base DiT module, dependency of S2V)
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit_mod
_spec.loader.exec_module(dit_mod)

# Load wan_video_dit_s2v.py (S2V model with audio encoding classes)
_s2v_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit_s2v.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit_s2v", _s2v_path)
s2v_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit_s2v"] = s2v_mod
_spec.loader.exec_module(s2v_mod)

from diffsynth.models.wan_video_dit_s2v import CausalConv1d, MotionEncoder_tc, CausalAudioEncoder
from einops import rearrange as rearr

print(f"Imported: CausalConv1d, MotionEncoder_tc, CausalAudioEncoder")
print("Setup complete.")

## Audio Encoding Pathway -- Complete Data Flow

```
Audio Encoding Pathway -- Complete Data Flow
=============================================

INPUT: WAV2Vec features [B, num_layers, audio_dim, video_length]
       (3 layers, dim=384, length=16 in reduced config)

STAGE 1 -- WEIGHTED AGGREGATION (CausalAudioEncoder):
  weights = SiLU(learnable_weights)       [1, 3, 1, 1]
  weighted_feat = sum(features * weights)  [B, 384, 16]
  permute(0, 2, 1)                         [B, 16, 384]

STAGE 2 -- MOTION ENCODER (MotionEncoder_tc):
  LOCAL PATH (multi-head, num_heads=2):
    conv1_local(384 -> 192)               [B, 192, 16]  (hidden_dim//4 * num_heads)
    rearrange to per-head                  [2, 16, 96]   (B*num_heads, T, hidden_dim//4)
    norm + SiLU                            [2, 16, 96]
    conv2(96 -> 192, stride=2)             [2, 192, 8]   (4x downsample: step 1)
    norm + SiLU                            [2, 8, 192]
    conv3(192 -> 384, stride=2)            [2, 384, 4]   (4x downsample: step 2)
    norm + SiLU                            [2, 4, 384]
    rearrange back                         [1, 4, 2, 384] (B, T//4, num_heads, dim)
    cat padding_token                      [1, 4, 3, 384] (B, T//4, num_heads+1, dim)

  GLOBAL PATH (single-head):
    conv1_global(384 -> 96)                [1, 96, 16]   (hidden_dim//4)
    norm + SiLU                            [1, 16, 96]
    conv2(96 -> 192, stride=2)             [1, 192, 8]
    norm + SiLU                            [1, 8, 192]
    conv3(192 -> 384, stride=2)            [1, 384, 4]
    norm + SiLU                            [1, 4, 384]
    final_linear(384 -> 384)               [1, 4, 384]
    rearrange                              [1, 4, 1, 384] (B, T//4, 1, dim)

OUTPUT:
  audio_emb_global: [1, 4, 1, 384]  (global features for AdaLayerNorm)
  audio_emb_local:  [1, 4, 3, 384]  (local features for CrossAttention)
                         ^
                    4 temporal frames = video_length(16) / 4x_downsample
```

In [ ]:
# Reduced config (consistent with all prior phases -- per D-09)
dim = 384
num_heads = 4
head_dim = dim // num_heads  # 96

# S2V-specific audio config (per D-08)
num_audio_token = 2           # reduced from production 4 (num_heads for MotionEncoder_tc)
audio_dim = 384               # matching dim for simplicity
num_wav2vec_layers = 3        # reduced from production 25
video_length = 16             # temporal length of audio input

# MotionEncoder_tc derived values (hidden_dim = dim = 384)
hidden_dim = dim              # CausalAudioEncoder passes out_dim=dim to MotionEncoder_tc

print(f"Reduced config: dim={dim}, num_heads={num_heads}, head_dim={head_dim}")
print(f"Audio config: num_audio_token={num_audio_token}, audio_dim={audio_dim}")
print(f"              num_wav2vec_layers={num_wav2vec_layers}, video_length={video_length}")
print(f"Derived: hidden_dim={hidden_dim}, hidden_dim//4={hidden_dim//4}")

## 1. CausalConv1d -- Causal Temporal Padding

CausalConv1d applies **asymmetric padding** `(kernel_size - 1, 0)` so each output timestep
only depends on the current and past inputs -- never the future. This is essential for
autoregressive or causal processing of temporal audio features.

Compare to **CausalConv3d from NB-08**: same concept, but CausalConv3d pads only along the
temporal dimension of 5D video tensors `[B, C, T, H, W]`, while CausalConv1d pads along
the single temporal dimension of 3D audio tensors `[B, C, T]`.

| Property | CausalConv1d (NB-13) | CausalConv3d (NB-08) |
|----------|---------------------|---------------------|
| Input shape | `[B, C, T]` | `[B, C, T, H, W]` |
| Causal padding | `(kernel_size-1, 0)` on T | `(kernel_size-1, 0, 0, 0, 0, 0)` on T |
| Spatial padding | N/A | Symmetric on H, W |
| Used in | Audio encoding (S2V) | VAE encoder/decoder |

In [ ]:
# Source: wan_video_dit_s2v.py, line 85
# Demonstrate causal padding: pad=(2, 0) for kernel_size=3
conv = CausalConv1d(chan_in=384, chan_out=192, kernel_size=3, stride=1)
print(f"CausalConv1d padding: {conv.time_causal_padding}")
assert conv.time_causal_padding == (2, 0), f"Expected (2, 0), got {conv.time_causal_padding}"

x = torch.randn(1, 384, 16)  # [B, C, T]
print(f"\nInput:  {x.shape}")

# Step 1: causal padding
x_padded = F.pad(x, conv.time_causal_padding, mode=conv.pad_mode)
print(f"Padded: {x_padded.shape}  (T + kernel_size - 1 = 16 + 2 = 18)")
assert x_padded.shape == torch.Size([1, 384, 18])

# Step 2: Conv1d
with torch.no_grad():
    y = conv(x)
print(f"Output: {y.shape}  (output length == input length for stride=1)")
assert y.shape == torch.Size([1, 192, 16]), f"Expected [1, 192, 16], got {y.shape}"
print(f"\nKey insight: causal padding preserves temporal length (16 -> 16)")

In [ ]:
# Source: wan_video_dit_s2v.py, line 114
# Stride=2 halves the temporal dimension
conv_s2 = CausalConv1d(chan_in=96, chan_out=192, kernel_size=3, stride=2)
x_s2 = torch.randn(1, 96, 16)
with torch.no_grad():
    y_s2 = conv_s2(x_s2)
print(f"Stride=2: {x_s2.shape} -> {y_s2.shape}")
assert y_s2.shape == torch.Size([1, 192, 8]), f"Expected [1, 192, 8], got {y_s2.shape}"
print(f"Temporal downsample: 16 -> 8 (halved by stride=2)")

# Also verify stride=2 with half the input length
x_s2b = torch.randn(1, 192, 8)
conv_s2b = CausalConv1d(chan_in=192, chan_out=384, kernel_size=3, stride=2)
with torch.no_grad():
    y_s2b = conv_s2b(x_s2b)
print(f"Stride=2: {x_s2b.shape} -> {y_s2b.shape}")
assert y_s2b.shape == torch.Size([1, 384, 4]), f"Expected [1, 384, 4], got {y_s2b.shape}"
print(f"Temporal downsample: 8 -> 4 (halved again)")
print(f"\nTwo stride-2 convolutions: 16 -> 8 -> 4 (total 4x downsample)")

## 2. MotionEncoder_tc -- Dual Local/Global Paths

MotionEncoder_tc is a dual-path architecture:

- **Local path** (multi-head): Produces fine-grained per-head audio features for cross-attention.
  Uses `conv1_local` which outputs `hidden_dim//4 * num_heads` channels, then splits into
  per-head streams that share `conv2` and `conv3` for temporal downsampling.

- **Global path** (single-head): Produces a single summary embedding per temporal frame for
  AdaLayerNorm modulation. Uses `conv1_global` which outputs only `hidden_dim//4` channels
  (no head multiplication), shares the same `conv2`/`conv3`/norms, then applies `final_linear`.

### Sub-Module Inventory

| Sub-module | Type | Input -> Output | Purpose |
|------------|------|----------------|--------|
| `conv1_local` | CausalConv1d | (384, 192, k=3, s=1) | Local path first layer (hidden_dim//4 * num_heads) |
| `conv1_global` | CausalConv1d | (384, 96, k=3, s=1) | Global path first layer (hidden_dim//4) |
| `norm1` | LayerNorm | (96) | Normalize after first conv (hidden_dim//4) |
| `act` | SiLU | -- | Activation after each norm |
| `conv2` | CausalConv1d | (96, 192, k=3, s=2) | Shared: temporal downsample 2x |
| `norm2` | LayerNorm | (192) | Normalize after conv2 |
| `conv3` | CausalConv1d | (192, 384, k=3, s=2) | Shared: temporal downsample 2x |
| `norm3` | LayerNorm | (384) | Normalize after conv3 |
| `final_linear` | Linear | (384, 384) | Global path: final projection |
| `padding_tokens` | Parameter | (1, 1, 1, 384) | Local path: extra token per frame |

In [ ]:
# Source: wan_video_dit_s2v.py, line 101
enc = MotionEncoder_tc(
    in_dim=audio_dim,       # 384
    hidden_dim=hidden_dim,  # 384
    num_heads=num_audio_token,  # 2
    need_global=True,
)
total_params = sum(p.numel() for p in enc.parameters())
print(f"MotionEncoder_tc: {total_params:,} parameters")
print(f"  in_dim={audio_dim}, hidden_dim={hidden_dim}, num_heads={num_audio_token}, need_global=True")

# Verify sub-module dimensions
assert enc.conv1_local.conv.in_channels == audio_dim  # 384
assert enc.conv1_local.conv.out_channels == hidden_dim // 4 * num_audio_token  # 96 * 2 = 192
print(f"\nconv1_local: {enc.conv1_local.conv.in_channels} -> {enc.conv1_local.conv.out_channels}")

assert enc.conv1_global.conv.in_channels == audio_dim  # 384
assert enc.conv1_global.conv.out_channels == hidden_dim // 4  # 96
print(f"conv1_global: {enc.conv1_global.conv.in_channels} -> {enc.conv1_global.conv.out_channels}")

assert enc.conv2.conv.in_channels == hidden_dim // 4  # 96
assert enc.conv2.conv.out_channels == hidden_dim // 2  # 192
assert enc.conv2.conv.stride == (2,)
print(f"conv2: {enc.conv2.conv.in_channels} -> {enc.conv2.conv.out_channels} (stride={enc.conv2.conv.stride})")

assert enc.conv3.conv.in_channels == hidden_dim // 2  # 192
assert enc.conv3.conv.out_channels == hidden_dim  # 384
assert enc.conv3.conv.stride == (2,)
print(f"conv3: {enc.conv3.conv.in_channels} -> {enc.conv3.conv.out_channels} (stride={enc.conv3.conv.stride})")

assert enc.padding_tokens.shape == torch.Size([1, 1, 1, hidden_dim])
print(f"padding_tokens: {enc.padding_tokens.shape}")

assert hasattr(enc, 'final_linear'), "need_global=True should create final_linear"
assert enc.final_linear.in_features == hidden_dim  # 384
assert enc.final_linear.out_features == hidden_dim  # 384
print(f"final_linear: {enc.final_linear.in_features} -> {enc.final_linear.out_features}")
print(f"\nAll sub-module dimensions verified.")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 125-146
# ---- LOCAL PATH (multi-head) ----
enc.eval()
x_audio = torch.randn(1, video_length, audio_dim)  # [B, T=16, C=384]
print(f"Input: {x_audio.shape}")

with torch.no_grad():
    # Step 1: rearrange to channels-first (matches line 126)
    x_loc = x_audio.permute(0, 2, 1)  # [B, C, T] = [1, 384, 16]
    x_ori = x_loc.clone()
    b, c, t = x_loc.shape
    print(f"\nStep 1 -- permute:     {x_loc.shape}")
    assert x_loc.shape == torch.Size([1, 384, 16])

    # Step 2: conv1_local (384 -> 192, no downsample)
    x_loc = enc.conv1_local(x_loc)
    print(f"Step 2 -- conv1_local: {x_loc.shape}  (384 -> hidden_dim//4 * num_heads = {hidden_dim//4 * num_audio_token})")
    assert x_loc.shape == torch.Size([1, 192, 16])

    # Step 3: rearrange to per-head (matches line 130)
    x_loc = rearr(x_loc, 'b (n c) t -> (b n) t c', n=num_audio_token)
    print(f"Step 3 -- per-head:    {x_loc.shape}  (B*num_heads, T, hidden_dim//4)")
    assert x_loc.shape == torch.Size([num_audio_token, 16, hidden_dim // 4])  # [2, 16, 96]

    # Step 4: norm1 + SiLU (matches lines 131-132)
    x_loc = enc.norm1(x_loc)
    x_loc = enc.act(x_loc)
    print(f"Step 4 -- norm+SiLU:   {x_loc.shape}")
    assert x_loc.shape == torch.Size([2, 16, 96])

    # Step 5: conv2 (stride=2, first 2x downsample) (matches lines 133-135)
    x_loc = rearr(x_loc, 'b t c -> b c t')
    x_loc = enc.conv2(x_loc)
    x_loc = rearr(x_loc, 'b c t -> b t c')
    print(f"Step 5 -- conv2(s=2):  {x_loc.shape}  (T: 16 -> 8)")
    assert x_loc.shape == torch.Size([2, 8, 192])

    # Step 6: norm2 + SiLU (matches lines 136-137)
    x_loc = enc.norm2(x_loc)
    x_loc = enc.act(x_loc)
    print(f"Step 6 -- norm+SiLU:   {x_loc.shape}")

    # Step 7: conv3 (stride=2, second 2x downsample -> total 4x) (matches lines 138-140)
    x_loc = rearr(x_loc, 'b t c -> b c t')
    x_loc = enc.conv3(x_loc)
    x_loc = rearr(x_loc, 'b c t -> b t c')
    print(f"Step 7 -- conv3(s=2):  {x_loc.shape}  (T: 8 -> 4, total 4x downsample)")
    assert x_loc.shape == torch.Size([2, 4, 384])

    # Step 8: norm3 + SiLU (matches lines 141-142)
    x_loc = enc.norm3(x_loc)
    x_loc = enc.act(x_loc)
    print(f"Step 8 -- norm+SiLU:   {x_loc.shape}")

    # Step 9: rearrange back to [B, T, num_heads, dim] (matches line 143)
    x_loc = rearr(x_loc, '(b n) t c -> b t n c', b=b)
    print(f"Step 9 -- rearrange:   {x_loc.shape}  (B, T//4, num_heads, dim)")
    assert x_loc.shape == torch.Size([1, 4, num_audio_token, hidden_dim])  # [1, 4, 2, 384]

    # Step 10: cat padding_tokens (matches lines 144-145)
    padding = enc.padding_tokens.repeat(b, x_loc.shape[1], 1, 1).to(device=x_loc.device, dtype=x_loc.dtype)
    x_local = torch.cat([x_loc, padding], dim=-2)
    print(f"Step 10 -- cat padding: {x_local.shape}  (num_heads+1 = {num_audio_token + 1})")
    assert x_local.shape == torch.Size([1, 4, num_audio_token + 1, hidden_dim])  # [1, 4, 3, 384]

print(f"\nLocal path output: {x_local.shape}  [B, T//4, num_audio_token+1, dim]")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 151-167
# ---- GLOBAL PATH (single-head) ----
# Key insight: global path does NOT multiply batch by num_heads.
# It processes x_ori (batch B=1) through conv1_global (single-head output),
# then reuses the SAME conv2/conv3/norms as the local path.
with torch.no_grad():
    x_glob = enc.conv1_global(x_ori)
    print(f"Step 1 -- conv1_global: {x_glob.shape}  (384 -> hidden_dim//4 = {hidden_dim//4})")
    assert x_glob.shape == torch.Size([1, hidden_dim // 4, 16])  # [1, 96, 16]

    # Same conv2/conv3 chain (shared modules) but on single-head batch
    x_glob = rearr(x_glob, 'b c t -> b t c')
    x_glob = enc.norm1(x_glob)
    x_glob = enc.act(x_glob)
    print(f"Step 2 -- norm+SiLU:    {x_glob.shape}")
    assert x_glob.shape == torch.Size([1, 16, 96])

    x_glob = rearr(x_glob, 'b t c -> b c t')
    x_glob = enc.conv2(x_glob)
    x_glob = rearr(x_glob, 'b c t -> b t c')
    x_glob = enc.norm2(x_glob)
    x_glob = enc.act(x_glob)
    print(f"Step 3 -- conv2(s=2):   {x_glob.shape}  (T: 16 -> 8)")
    assert x_glob.shape == torch.Size([1, 8, 192])

    x_glob = rearr(x_glob, 'b t c -> b c t')
    x_glob = enc.conv3(x_glob)
    x_glob = rearr(x_glob, 'b c t -> b t c')
    x_glob = enc.norm3(x_glob)
    x_glob = enc.act(x_glob)
    print(f"Step 4 -- conv3(s=2):   {x_glob.shape}  (T: 8 -> 4)")
    assert x_glob.shape == torch.Size([1, 4, 384])

    x_glob = enc.final_linear(x_glob)
    print(f"Step 5 -- final_linear: {x_glob.shape}")
    assert x_glob.shape == torch.Size([1, 4, 384])

    # rearrange with b=b (global path batch is B=1, not B*num_heads)
    # n resolves to 1 since batch dim was never multiplied by num_heads
    x_global = rearr(x_glob, '(b n) t c -> b t n c', b=b)
    print(f"Step 6 -- rearrange:    {x_global.shape}  (n=1 because global path is single-head)")
    assert x_global.shape == torch.Size([1, 4, 1, 384])

print(f"\nGlobal path output: {x_global.shape}  [B, T//4, 1, dim]")
print(f"\nAsymmetry: local has {num_audio_token+1} tokens/frame, global has 1 token/frame")

In [ ]:
# Source: wan_video_dit_s2v.py, line 125
# Verify: direct call produces same (global, local) as manual trace
with torch.no_grad():
    result = enc(x_audio)
    global_direct, local_direct = result

assert global_direct.shape == x_global.shape, f"Global shape mismatch: {global_direct.shape} vs {x_global.shape}"
assert local_direct.shape == x_local.shape, f"Local shape mismatch: {local_direct.shape} vs {x_local.shape}"
assert torch.allclose(global_direct, x_global, atol=1e-5), "Global path: manual trace must match direct call"
assert torch.allclose(local_direct, x_local, atol=1e-5), "Local path: manual trace must match direct call"
print(f"Verification PASSED: direct call matches manual trace")
print(f"  global: {global_direct.shape}  local: {local_direct.shape}")

## 3. CausalAudioEncoder -- Weighted Layer Aggregation

CausalAudioEncoder sits on top of MotionEncoder_tc. It takes multi-layer WAV2Vec features
(each layer captures different audio characteristics), compresses them via learnable
SiLU-gated weighted aggregation, then routes the result through MotionEncoder_tc.

The key innovation: instead of using a single WAV2Vec layer, it learns which layers
matter most via differentiable weights initialized at 0.01 and gated through SiLU.

In [ ]:
# Source: wan_video_dit_s2v.py, line 321
audio_enc = CausalAudioEncoder(
    dim=audio_dim,                   # 384 (per D-08)
    num_layers=num_wav2vec_layers,    # 3 (reduced from production 25)
    out_dim=dim,                      # 384
    num_token=num_audio_token,        # 2 (per D-08)
    need_global=True,                 # needed for AdaLayerNorm in AudioInjector
)
total_params = sum(p.numel() for p in audio_enc.parameters())
print(f"CausalAudioEncoder: {total_params:,} parameters")

# Inspect learnable weights
print(f"\nWeights shape: {audio_enc.weights.shape}  (1, num_layers, 1, 1)")
assert audio_enc.weights.shape == torch.Size([1, num_wav2vec_layers, 1, 1])
print(f"Initial weight values: {audio_enc.weights.data.squeeze().tolist()}")
print(f"After SiLU gating: {audio_enc.act(audio_enc.weights.data).squeeze().tolist()}")
print(f"\nKey: weights start at 0.01 -- nearly uniform. Training adjusts which layers matter.")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 331-338
B_aud = 1
features = torch.randn(B_aud, num_wav2vec_layers, audio_dim, video_length)
print(f"Input features: {features.shape}  [B, num_layers, dim, video_length]")

audio_enc.eval()
with torch.no_grad():
    # Step 1: SiLU-gated weights
    weights = audio_enc.act(audio_enc.weights.to(device=features.device, dtype=features.dtype))
    print(f"\nStep 1 -- SiLU(weights):       {weights.shape}")
    assert weights.shape == torch.Size([1, num_wav2vec_layers, 1, 1])

    # Step 2: Normalize weights (sum over layers for weighted average)
    weights_sum = weights.sum(dim=1, keepdims=True)
    print(f"Step 2 -- weights_sum:          {weights_sum.shape}  (sum over layers)")

    # Step 3: Weighted aggregation -- collapse num_layers dimension
    weighted_feat = ((features * weights) / weights_sum).sum(dim=1)
    print(f"Step 3 -- weighted_feat:        {weighted_feat.shape}  (collapsed {num_wav2vec_layers} layers -> 1)")
    assert weighted_feat.shape == torch.Size([B_aud, audio_dim, video_length])  # [1, 384, 16]

    # Step 4: Permute to [B, T, C] for MotionEncoder_tc
    weighted_feat = weighted_feat.permute(0, 2, 1)
    print(f"Step 4 -- permute(0,2,1):       {weighted_feat.shape}")
    assert weighted_feat.shape == torch.Size([B_aud, video_length, audio_dim])  # [1, 16, 384]

    # Step 5: Route through MotionEncoder_tc
    result = audio_enc.encoder(weighted_feat)
    audio_global, audio_local = result
    print(f"Step 5 -- MotionEncoder_tc:")
    print(f"  global: {audio_global.shape}")
    print(f"  local:  {audio_local.shape}")
    assert audio_global.shape == torch.Size([1, 4, 1, 384])
    assert audio_local.shape == torch.Size([1, 4, num_audio_token + 1, 384])  # [1, 4, 3, 384]

print(f"\nFinal: global {audio_global.shape}, local {audio_local.shape}")
print(f"4 temporal frames = video_length({video_length}) / 4x_downsample")

In [ ]:
# Source: wan_video_dit_s2v.py, line 331
# Verify: direct CausalAudioEncoder call matches step-by-step trace
with torch.no_grad():
    direct_result = audio_enc(features)
    direct_global, direct_local = direct_result

assert torch.allclose(direct_global, audio_global, atol=1e-5), "Global: direct must match manual"
assert torch.allclose(direct_local, audio_local, atol=1e-5), "Local: direct must match manual"
print(f"Verification PASSED: CausalAudioEncoder direct call matches step-by-step trace")
print(f"  global: {direct_global.shape}  local: {direct_local.shape}")

## Summary

### Key Takeaways

1. **CausalConv1d pads only the past**, preserving causality -- output length equals input length for stride=1. Two stride-2 CausalConv1d layers produce the 4x temporal downsampling used by MotionEncoder_tc.
2. **MotionEncoder_tc has asymmetric local (multi-head) and global (single-head) paths**, both with 4x temporal downsampling via two stride-2 convolutions. The local path produces `num_heads+1` tokens per temporal frame (including a learnable padding token), while the global path produces exactly 1 token per frame.
3. **CausalAudioEncoder learns to weight WAV2Vec layers** via SiLU-gated aggregation (initialized at 0.01 for near-uniform weighting), then routes the result through MotionEncoder_tc to produce the final `(global, local)` audio embeddings.

### Source References

| Symbol | Location |
|--------|----------|
| CausalConv1d | `wan_video_dit_s2v.py`, line 85 |
| MotionEncoder_tc | `wan_video_dit_s2v.py`, line 101 |
| MotionEncoder_tc.forward | `wan_video_dit_s2v.py`, line 125 |
| CausalAudioEncoder | `wan_video_dit_s2v.py`, line 321 |
| CausalAudioEncoder.forward | `wan_video_dit_s2v.py`, line 331 |

### What's Next

- **NB-14** covers FramePackMotioner and per-token RoPE -- how motion latents from multiple temporal resolutions are patchified and given position embeddings.
- **NB-15** shows how these audio embeddings are injected into the DiT via AudioInjector -- the global embedding modulates via AdaLayerNorm and the local embeddings are consumed via cross-attention.

## Exercises

### Exercise 1 -- Change num_audio_token

Change `num_audio_token` from 2 to 4. How does this affect the local path output shape? What stays the same in the global path? Predict before running.

**Hint:** The local path output is `[B, T//4, num_audio_token+1, dim]`, so changing `num_audio_token` changes the third dimension. The global path is single-head and unaffected.

### Exercise 2 -- The padding_tokens Parameter

The `padding_tokens` parameter adds 1 extra token per temporal frame in the local path. Why might this be useful?

**Hint:** It gives the model a learnable 'null' audio token that can be attended to when no audio information is relevant -- similar to how language models use a `[PAD]` token. The AudioInjector's cross-attention can learn to attend to this padding token for video frames where audio is less informative.

### Exercise 3 -- Temporal downsampling calculation

Compute the total temporal downsampling factor by multiplying the strides of conv2 and conv3 (both stride=2). What `video_length` would produce exactly 1 output temporal frame?

**Hint:** Total downsample = 2 x 2 = 4x. For 1 output frame, you need `video_length = 4`. Try `torch.randn(1, 4, 384)` as input to verify.